# batchnorm-running-stats — worked example 2: Eval-mode output is invariant to the input batch's own stats

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `batchnorm-running-stats`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In eval mode, BatchNorm freezes its buffers: it normalizes every input using the stored `running_mean` / `running_var` and never looks at the current batch's statistics. A direct consequence is that scaling or shifting which samples are in the eval batch does not change how any individual sample is normalized — the transform is a fixed affine map per channel. This is exactly what makes inference deterministic and batch-size independent.

## Worked solution

**Goal.** Show that in `.eval()` mode, normalizing a single sample alone gives the same result as normalizing it as part of a larger, differently-distributed batch.

**Step 1 — build a BN with known buffers.** We create a `BatchNorm2d`, then overwrite `running_mean` and `running_var` with non-trivial values so the eval transform is clearly not the identity. We leave `weight=1`, `bias=0` so we isolate the standardization step.

**Step 2 — switch to eval.** `bn.eval()` flips the internal flag so the forward pass uses the buffers, not batch stats, and stops updating them.

**Step 3 — normalize the sample two ways.** First we pass one sample `x0` alone. Then we concatenate `x0` with three garbage samples drawn from a totally different distribution and pass the whole batch. We pull `x0`'s row back out of the output.

**Step 4 — compare.** The two outputs for `x0` are identical (up to float tolerance). Why: eval normalizes as `(x - running_mean) / sqrt(running_var + eps)`, a per-channel affine map with NO dependence on the other rows. In train mode this would fail, because the batch mean/var would shift when we add the garbage samples.

In [ ]:
bn = t.nn.BatchNorm2d(3)
with t.no_grad():
    bn.running_mean.copy_(t.tensor([0.5, -1.0, 2.0]))
    bn.running_var.copy_(t.tensor([2.0, 0.5, 4.0]))
bn.eval()

t.manual_seed(0)
x0 = t.randn(1, 3, 4, 4)
garbage = t.randn(3, 3, 4, 4) * 10 + 50

out_alone = bn(x0)
out_in_batch = bn(t.cat([x0, garbage], dim=0))[0:1]

max_diff = (out_alone - out_in_batch).abs().max().item()
print("max abs difference for x0:", round(max_diff, 8))
print("identical within tol:", t.allclose(out_alone, out_in_batch, atol=1e-6))